#### 1. Configurations

> **Tokens are optional.** Public HuggingFace models and public CivitAI resources download fine with both fields blank - tokens only matter for **gated/private** HF models (and the CivitAI field only feeds its metadata lookup, not the download itself). Paste them into the **HuggingFace token** / **CivitAI token** fields below (or set the `HF_TOKEN` / `CIVITAI_TOKEN` environment variables) if you need gated models. They are saved in `links.json`, so you only enter them once.
> - HuggingFace read token: https://huggingface.co/settings/tokens
> - CivitAI API key: https://civitai.com/user/account (API Keys)

In [ ]:
# Run to initialize and set parameters    . 
import site, sys, time, json, glob, zipfile, tarfile, re, time, shutil, gdown, subprocess, urllib, requests
from urllib.parse import urlparse, parse_qs, unquote
from typing import List, Tuple, Optional, Any
from ipywidgets import Checkbox, Text, Password, Button, HBox, ToggleButton, Label, Box, Layout, Accordion, VBox, CallbackDispatcher
from IPython.display import display, HTML, clear_output
from pathlib import Path
from subprocess import check_output, check_call
from tempfile import NamedTemporaryFile
from os import getenv, environ


display("Starting ....")

# Hardcoded - this notebook is built around this one maintainer-uploaded
# package (see build_package.py / build_package.ipynb), not a generic field
# users are expected to fill in.
OFFLINE_PACKAGE_URL = "https://huggingface.co/datasets/Xakiru/paperspace-sdw/resolve/main/sdw_package.tar.xz"

# Path inside the offline package (see build_package.py's EXTRA_DIR) where the
# maintainer's tuned config.json is bundled alongside the webui tree, not inside it.
EXTRA_DIR = "_package_extra"

if not (shutil.which("aria2c") and shutil.which("lz4")):
    !sudo apt -qq update 
    !sudo apt install -qy  --fix-missing libcairo2-dev aria2 lz4
clear_output(wait=True)




# paperspace-provided directories
notebook_dir = Path("/notebooks")
# local-only semi-persistent storage
content_dir = Path("/content")

sdw_dir = content_dir.joinpath("sdw")
models_dir = sdw_dir.joinpath("models")
extensions_dir = sdw_dir.joinpath("extensions")
output_dir = notebook_dir.joinpath("outputs/") 
cached_dir = notebook_dir.joinpath("cached")
if not cached_dir.exists():
    cached_dir.mkdir(exist_ok=True)
    
# Mapping resources folders 
paths = {
    "models": "models/Stable-diffusion",
    "loras": "models/Lora",
    "vaes": "models/VAE",
    "embeddings": "embeddings",
    "extensions":  "extensions",  
}
# options for aria2 to use
# Single connection (-x1 -s1) is the safe default: several hosts (CivitAI's B2
# storage, HuggingFace's newer Xet CDN) hand out a freshly-signed, single-use
# redirect URL per request, and segmented downloads (-x16 -s16) hit that URL
# many times in parallel, so all but one segment gets rejected with a 403.
# The "Fast parallel downloads" checkbox below opts back into -x16 -s16 for
# people who want the speed and are fine retrying on the occasional 403.
def get_aria2_opts() -> list:
    base = ["--console-log-level=error", "--summary-interval=30", "-j1", "-k1M", "-c"]
    try:
        fast = bool(fast_download_control.value)
    except Exception:
        fast = False
    return base + (["-x16", "-s16"] if fast else ["-x1", "-s1"])

# ---- HuggingFace / CivitAI authentication ----
# Tokens come from the UI fields below (persisted in links.json) or the env
# vars HF_TOKEN / CIVITAI_TOKEN. HuggingFace uses an Authorization header;
# CivitAI's token is only used (as a header) for the v1/model-versions metadata
# lookup in civitai_meta() - see authorize_url() for why it's NOT put on the
# actual download URL.
def get_token(kind: str) -> str:
    val = ""
    try:
        if kind == "hf":
            val = (hf_token_control.value or "").strip()
        else:
            val = (civitai_token_control.value or "").strip()
    except Exception:
        val = ""
    if val:
        return val
    if kind == "hf":
        return (getenv("HF_TOKEN") or getenv("HUGGING_FACE_HUB_TOKEN") or "").strip()
    return (getenv("CIVITAI_TOKEN") or "").strip()

def authorize_url(url: str) -> str:
    # Used to append ?token=<civitai_token> to the actual CivitAI download URL.
    # Removed: every attempt with the token attached 403'd at B2 even with a
    # single connection, no resume, a browser User-Agent and a Referer header -
    # while the v1/model-versions metadata lookup (token sent as an Authorization
    # header, not a query param) succeeded every time. Iyashinouta/sd-model-downloader
    # (a known-working reference) never sends a CivitAI token at all and works fine.
    # So: hit the bare download URL, same as that reference, and only use the
    # token where it's proven to work (the metadata header). This will break
    # truly gated/login-required CivitAI resources - revisit if that comes up.
    return url

def auth_headers(url: str) -> dict:
    headers = {}
    if "huggingface.co" in url:
        token = get_token("hf")
        if token:
            headers["Authorization"] = f"Bearer {token}"
    if "civitai.com" in url:
        # hotlink/Referer protection on CivitAI's CDN - see BROWSER_HEADERS
        headers["Referer"] = "https://civitai.com/"
    return headers

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://civitai.com/",
}

def civitai_requests_download(url: str, dst_path) -> bool:
    # aria2 got a 403 on this exact URL every time - single connection, no
    # resume, browser User-Agent, Referer, with or without a token - while
    # clicking the same link in a real browser downloads fine. That combination
    # means the block isn't about headers at all: it's almost certainly a
    # TLS/HTTP client fingerprint check (Cloudflare-style bot detection looking
    # at the JA3 handshake / HTTP version negotiation), which aria2's own TLS
    # stack can't be made to pass no matter what headers it sends. Switching to
    # Python's `requests` (used by the known-working Iyashinouta/sd-model-downloader
    # reference too) gives a different, more "normal" client fingerprint.
    dst_path = Path(dst_path)
    try:
        with requests.get(url, headers=BROWSER_HEADERS, stream=True, timeout=60, allow_redirects=True) as resp:
            resp.raise_for_status()
            tmp_path = dst_path.with_suffix(dst_path.suffix + ".part")
            with open(tmp_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
            tmp_path.replace(dst_path)
        if looks_like_failed_download(dst_path):
            display(f"[download FAILED] {dst_path.name} looks like an error/login page - check your token. Removing it.")
            dst_path.unlink()
            return False
        return True
    except Exception as e:
        display(f"[civitai] download failed for {dst_path.name}: {e}")
        tmp_path = dst_path.with_suffix(dst_path.suffix + ".part")
        if tmp_path.exists():
            tmp_path.unlink()
        return False

def looks_like_failed_download(path) -> bool:
    # True if a downloaded file is empty or an HTML/JSON error or login page
    try:
        p = Path(path)
        if not p.exists() or not p.is_file():
            return False
        if p.stat().st_size == 0:
            return True
        with open(p, "rb") as _fh:
            head = _fh.read(256).lstrip().lower()
        if head.startswith(b"<!doctype html") or head.startswith(b"<html"):
            return True
        if head.startswith(b"{") and b"error" in head:
            return True
        return False
    except Exception:
        return False

def civitai_meta(url):
    # CivitAI public API -> (filename, image_url) for a download URL; ("", None) on failure
    import urllib.request
    try:
        version_id = urlparse(url).path.rstrip("/").split("/")[-1]
        api = f"https://civitai.com/api/v1/model-versions/{version_id}"
        headers = {"User-Agent": "Mozilla/5.0"}
        token = get_token("civitai")
        if token:
            headers["Authorization"] = f"Bearer {token}"
        req = urllib.request.Request(api, headers=headers)
        data = json.loads(urllib.request.urlopen(req, timeout=20).read().decode())
        files = data.get("files", [])
        f = next((x for x in files if x.get("primary")), files[0] if files else None)
        fname = ((f or {}).get("name") or "").replace(" ", "_")
        img = next((im.get("url") for im in data.get("images", []) if im.get("type", "image") == "image"), None)
        return fname, img
    except Exception as e:
        display(f"[civitai] metadata lookup failed: {e}")
        return "", None


def _to_path(x):
    return x if isinstance(x, Path) else Path(x)

def inject(src, dst, isDirectory=False):
    """
    Unified injection system.

    isDirectory=False  → inject individual files (safe overwrite)
    isDirectory=True   → replace destination with a folder-level symlink
    """
    try:
        src = _to_path(src).expanduser().resolve()
        # dst is intentionally NOT .resolve()'d: if dst is already a symlink
        # (e.g. from a previous run), resolve() follows it straight through to
        # src's real path, so the is_symlink()/is_dir() checks below would be
        # checking src itself instead of dst - making an already-correct
        # symlink look like a pre-existing real directory and falsely refusing
        # to "overwrite" it every subsequent run.
        dst = _to_path(dst).expanduser()

        # Always ensure src exists
        src.mkdir(parents=True, exist_ok=True)

        # -----------------------------------------
        # FOLDER-LEVEL MODE (equivalent to expose)
        # -----------------------------------------
        if isDirectory:
            # Clean dst if it's a file or a symlink
            if dst.exists() or dst.is_symlink():
                if dst.is_symlink() or dst.is_file():
                    dst.unlink()
                elif dst.is_dir():
                    print(f"[inject DIR ERROR] '{dst}' is a directory. Refusing to overwrite.")
                    return False

            # Ensure parent folder exists
            dst.parent.mkdir(parents=True, exist_ok=True)

            dst.symlink_to(src, target_is_directory=True)
            print(f"[inject DIR OK] {dst} → {src}")
            return True

        # -----------------------------------------
        # FILE-LEVEL MODE
        # -----------------------------------------
        # Ensure dst exists
        dst.mkdir(parents=True, exist_ok=True)

        # Validate
        if not src.is_dir():
            print(f"[inject ERROR] Source is not a folder → {src}")
            return False

        if not dst.is_dir():
            print(f"[inject ERROR] Destination is not a folder → {dst}")
            return False

        # Inject files from src into dst
        for file in src.iterdir():
            if file.is_file():
                dst_file = dst / file.name

                if dst_file.exists() or dst_file.is_symlink():
                    try:
                        dst_file.unlink()
                    except IsADirectoryError:
                        print(f"[inject ERROR] '{dst_file}' is a directory. Skipped.")
                        continue

                dst_file.symlink_to(file)
                print(f"[inject OK] {dst_file} → {file}")

        return True

    except Exception as e:
        print(f"[inject EXCEPTION] {e}")
        return False

def extract_offline_package(source, dst_dir) -> bool:
    # Seeds sdw_dir from a prebuilt archive (webui + its 5 pinned sub-repos + a chosen
    # set of extensions, see build_package.py) instead of doing ~11 separate git clones.
    # .git directories are kept intact inside the archive, so this is purely a fast path:
    # webui's own git_clone() sees each sub-repo already at the expected commit and skips
    # re-cloning it, and the existing "Update stable diffusion first" / "Update all
    # extensions on launch" checkboxes still work normally (git pull) on top of this seed.
    # Archive is a tar.xz (build_package.py's pack_package()) - .zip is also accepted as
    # a fallback in case an older package or a manually-zipped archive is used instead.
    #
    # `source` is either a URL (downloaded to a temp file then removed) or a local
    # Path that already exists on disk (used in place, never deleted) - see the
    # "Local package file" field below, which lets the package be picked up from
    # next to the notebook instead of fetched over the network every time.
    dst_dir = Path(dst_dir)
    is_url = str(source).lower().startswith(("http://", "https://"))
    is_zip = str(source).lower().split("?")[0].endswith(".zip")
    downloaded_path = None
    try:
        if is_url:
            downloaded_path = dst_dir.parent / ("_offline_package.zip" if is_zip else "_offline_package.tar.xz")
            archive_path = downloaded_path
            # dst_dir.parent (e.g. /content) isn't guaranteed to exist yet on a
            # fresh instance - writing the archive there before creating it fails
            # with FileNotFoundError, silently falling back to git clone.
            archive_path.parent.mkdir(parents=True, exist_ok=True)
            display(f"Downloading offline package from {source} ...")
            with requests.get(source, headers=BROWSER_HEADERS, stream=True, timeout=120, allow_redirects=True) as resp:
                resp.raise_for_status()
                with open(archive_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
        else:
            archive_path = Path(source)
            if not archive_path.exists():
                display(f"[offline package] local file not found: {archive_path}")
                return False
            display(f"Using local offline package: {archive_path} ...")
        dst_dir.mkdir(parents=True, exist_ok=True)
        if is_zip:
            with zipfile.ZipFile(archive_path) as zf:
                zf.extractall(dst_dir)
        else:
            with tarfile.open(archive_path, "r:xz") as tf:
                tf.extractall(dst_dir)

        # Pull the bundled config.json (packed under EXTRA_DIR, a sibling path -
        # see build_package.py) out to the persistent notebook root, then drop the
        # now-empty EXTRA_DIR so it doesn't linger inside sdw_dir. Skipped if a
        # config.json is already there - don't clobber a user's own customizations.
        extra_dir = dst_dir / EXTRA_DIR
        bundled_config = extra_dir / "config.json"
        if bundled_config.exists():
            dest_config = notebook_dir / "config.json"
            if not dest_config.exists():
                shutil.move(str(bundled_config), str(dest_config))
                display(f"[offline package] seeded {dest_config} from the bundled config.")
            shutil.rmtree(extra_dir, ignore_errors=True)

        return True
    except Exception as e:
        display(f"[offline package] failed, falling back to git clone: {e}")
        shutil.rmtree(dst_dir, ignore_errors=True)
        return False
    finally:
        if downloaded_path is not None and downloaded_path.exists():
            downloaded_path.unlink()

# func to download resources
def acquire_checkpoints(checkpoint_urls: list[str], fldr: str = "Stable-diffusion", is_cached: bool = False) -> None:

    sdw_fldr=paths.get(fldr, fldr)
    dst=sdw_dir.joinpath(sdw_fldr)
    real_dst=sdw_dir.joinpath(sdw_fldr)
    
    
    # cleaning symlinks and cache uncached
    if(is_cached) :
        #cleaning symbolic links
        sym_dst=sdw_dir.joinpath(paths.get(fldr, fldr))
        if sym_dst.exists() :
            delete_symlinks(sym_dst)
        #cleaning cached
        dst=cached_dir.joinpath(fldr)
        if not dst.exists():
            dst.mkdir(exist_ok=True)
        else :
            for item in dst.iterdir(): 
                if item.name.startswith(".") :
                    continue
                url=params.link_from_name(item.name)
                enabled, keep=params.get_link_status(url)
                if not keep and not enabled:
                    #delete from cache
                    item.unlink()
                    
                elif not keep and enabled:
                    # move away from cache
                    moved_dst=real_dst.joinpath(item.name)
                    if not moved_dst.exists() :
                         shutil.move(item, real_dst)
                    else :
                        item.unlink()               
        if real_dst.exists():
            for item in real_dst.iterdir(): 
                if item.name.startswith(".") :
                    continue
                url=params.link_from_name(item.name)
                enabled, keep=params.get_link_status(url)
                if  keep and enabled:
                    # move away from cache
                    moved_dst=dst.joinpath(item.name)
                    if not moved_dst.exists() :
                        shutil.move(item,dst)
                    else :
                        item.unlink()        
                                  
    if len(checkpoint_urls) == 0 :
        return  
    aria2_task_lines = []
    for url in checkpoint_urls:
        filename=params.name_from_link(url)
        if fldr=="extensions" :
            if filename == "" :
                filename=urlparse(url).path.split('/')[-1]
                filename=filename.replace('.git','')
            if not dst.joinpath(filename).exists():
                !git clone --quiet {url} {dst}/{filename} 
            else:
                !cd {dst}/{filename} && git fetch --quiet > /dev/null 2>&1 && git merge --quiet
        if 'civitai.com' in url:
            already = bool(filename) and dst.joinpath(filename).exists()
            if not already:
                civ_name, civ_img = civitai_meta(url)
                if civ_name:
                    filename = civ_name
                    params.set_link_name(url, filename)
                if not filename:
                    filename = wget_name(url, dst)
                    if filename:
                        params.set_link_name(url, filename)
                elif not dst.joinpath(filename).exists():
                    # Downloaded directly via requests, not queued through aria2 -
                    # see civitai_requests_download() for why.
                    if civitai_requests_download(url, dst.joinpath(filename)):
                        if civ_img:
                            _base = Path(filename).stem
                            _imgext = Path(urlparse(civ_img).path).suffix or ".jpeg"
                            _imgname = f"{_base}.preview{_imgext}"
                            if not dst.joinpath(_imgname).exists():
                                civitai_requests_download(civ_img, dst.joinpath(_imgname))
        elif 'drive.google' in url:
            if not dst.joinpath(filename).exists():
                downloaded_file_path = gdown.download(url, str(dst)+"/", quiet=True, fuzzy=True) 
                filename = Path(downloaded_file_path).name
                params.set_link_name(url,filename)
        elif filename=="" :
            filename= wget_name(url,dst)
            if filename :
                params.set_link_name(url,filename)
        elif "login" not in filename:
            if not dst.joinpath(filename).exists():
                aria2_task_lines.append(authorize_url(url))
                aria2_task_lines.append(f"\tout={filename}")
                for _hk, _hv in auth_headers(url).items():
                    aria2_task_lines.append(f"\theader={_hk}: {_hv}")
            params.set_link_name(url,filename)
          
    if len(aria2_task_lines) > 0:
        with NamedTemporaryFile("w") as aria2_file:
            aria2_file.write("\n".join(aria2_task_lines))
            aria2_file.flush()
            try:
                check_call(["aria2c", f"--input-file={aria2_file.name}"] + get_aria2_opts(),cwd=dst,)
            except Exception as e:
                display(f"[{fldr}] aria2 download(s) failed, continuing with the rest: {e}")
        # safety: flag/remove any file that came back as an error/login page
        for _ln in aria2_task_lines:
            if _ln.startswith("\tout="):
                _name = _ln[len("\tout="):]
                _fp = dst.joinpath(_name)
                if looks_like_failed_download(_fp):
                    display(f"[download FAILED] {_name} looks like an error/login page - check your token. Removing it.")
                    try:
                        _fp.unlink()
                    except Exception:
                        pass
            
    if is_cached :
        for file in dst.iterdir():
            # hidden files
            if file.name.startswith('.') or file.name.endswith('.aria2') :  
                continue

            #extensions
            if fldr =="extensions":
                extension_symlink = Path(real_dst / file.name) 
                if not all(
                    (
                        extension_symlink.is_symlink(),
                        extension_symlink.resolve() == file,
                    )
                ):
                    # recreate the symlink
                    extension_symlink.unlink(missing_ok=True)
                    extension_symlink.symlink_to(file, target_is_directory=True)
                continue

            original_file = real_dst / file.name      # symlink location (destination)
            symlink_target = file                     # real cached file (source)

            # Ensure destination directory exists
            original_file.parent.mkdir(parents=True, exist_ok=True)

            # Remove existing file or symlink
            if original_file.exists() or original_file.is_symlink():
                #display(f"Removing existing file or symlink: {original_file}")
                original_file.unlink()
            try:
                original_file.symlink_to(symlink_target)
                #display(f"Created symbolic link: {original_file} -> {symlink_target}")
            except Exception as e:
                display(f"Failed to create symbolic link: {e}") 


def filename_wget(url :str, timeout : int=5) -> Optional[str]:

    #batchlinks-downloader method
    dl_url = authorize_url(url)
    _req_headers = {'User-Agent': 'Mozilla/5.0'}
    _req_headers.update(auth_headers(url))
    req = urllib.request.Request(dl_url, headers=_req_headers)
    response = urllib.request.urlopen(req, timeout=timeout)
    forward_url = response.geturl()

    match = re.search(r'filename%3D%22(.+?)%22', forward_url)
    if match:
        filename = urllib.parse.unquote(match.group(1))
        return filename

    #wget method
    dst=Path('/notebooks/.tmp')
    if not dst.exists():
        dst.mkdir()
        
    _wget_cmd = ['wget', '--content-disposition', '-P', str(dst)]
    for _hk, _hv in auth_headers(url).items():
        _wget_cmd += ['--header', f'{_hk}: {_hv}']
    _wget_cmd.append(dl_url)
    process = subprocess.Popen(_wget_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    filename = None
    start_time = time.time()

    while True:
        if time.time() - start_time > timeout:
            display("Timeout reached.")
            break
        line = process.stdout.readline()
        if line == '' and process.poll() is not None:
            break
        if 'Saving to:' in line:
            match = re.search(r'Saving to: ‘.+/(.+)’', line)
            if match:
                filename = match.group(1)
                break
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=0.5)
        except subprocess.TimeoutExpired:
            process.kill()
    if filename:
        file = dst.joinpath(filename)
        if file.exists() :
            file.unlink()
    return filename
    
def delete_symlinks(directory_path:str)  -> None :
    directory = Path(directory_path)
    if not directory.exists() or not directory.is_dir():
        return
    for item in directory.iterdir():  
        if item.is_symlink():
            item.unlink() 
 
# Folder size method 
def get_directory_size(directory_pattern:str) -> str:
    try:
        if '*' in directory_pattern:
            directories = glob.glob(directory_pattern)
        else:
            directory_pattern_path = Path(directory_pattern)
            directories = [directory_pattern] if directory_pattern_path.is_dir() else []
        if not directories:
            return "0B"
        total_size = 0
        for directory in directories:
            output = check_output(['du', '-s', directory], text=True)
            size = int(output.split()[0])  # Size in blocks
            total_size += size * 1024  # Convert to bytes (assuming block size is 1024 bytes)

        # Convert bytes to a human-readable format
        for unit in ['B', 'KB', 'MB', 'GB', 'TB', 'PB']:
            if total_size < 1024:
                return f"{total_size:.2f}{unit}"
            total_size /= 1024

        return f"{total_size:.2f}PB"
    except Exception as e:
        #print("An error occurred:", e)
        return "0B"

# copy dep folder
def copy_recent_files(src_dir:str, dst_dir:str, minutes:int=10) -> None:
    # Calculate the cutoff time
    cutoff_time = time.time() - minutes * 60 *1000
    
    # Ensure the destination directory exists
    dst_dir_path = Path(dst_dir)
    dst_dir_path.mkdir(parents=True, exist_ok=True)

    # Iterate through the immediate files in src_dir
    for item in Path(src_dir).iterdir():
        if item.stat().st_mtime >= cutoff_time:
            if item.is_dir():
                # Use shutil.copytree for directories
                shutil.copytree(
                    item,
                    dst_dir_path / item.name,
                    symlinks=True,
                    ignore=shutil.ignore_patterns("*.pyc"),
                    dirs_exist_ok=True,
                )
            elif item.is_file() and not item.name.endswith('.pyc'):
                # Use shutil.copy2 for files
                shutil.copy2(item, dst_dir_path / item.name)
                




def wget_name(url:str,dst:str) -> Optional[str]:
    # CivitAI token goes in the URL; HuggingFace token goes in an Authorization header
    dl_url = authorize_url(url)
    header_args = " ".join([f'--header="{k}: {v}"' for k, v in auth_headers(url).items()])
    # Running wget command to get headers (and download the file)
    output = !wget -S -P {dst} --content-disposition {header_args} {dl_url}
    # Process the output to extract the filename
    output_str = "\n".join(output)
    # Use regular expression to find the filename
    match = re.search(r'Content-Disposition:.*filename="(.+)"', output_str, re.IGNORECASE)

    if match :
        new_file=f"{dst}/{match.group(1)}"
        #cleaning up the uwanted duplicates if there is any
        !rm -rf {new_file}.*
        if looks_like_failed_download(new_file):
            display(f"[download FAILED] {match.group(1)} looks like an error/login page - check your token. Removing it.")
            try:
                Path(new_file).unlink()
            except Exception:
                pass
            return None
        return match.group(1)
    return None


class Params:
    # default parameters
    file_path=""
    config_file="config.json"
    controls =[]
    accordions ={}
    # Used as the seed for links.json when no links.json exists at the
    # notebook root yet (Params.load() calls save() in that case) - this is
    # the "default" set a genuinely fresh instance gets, independent of
    # whatever the offline package contains.
    links={
        "models": ["https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1B_orangemixs.safetensors"],
        "vaes":[
            "https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt",
        ],
        "loras": [],
        "extensions": [],
        "embeddings": [],
        "ui": {"params": {},"enabled": [
            "https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1B_orangemixs.safetensors",
            "https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt",
        ],"keep": [],"names": {"https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1B_orangemixs.safetensors": "AOM3A1B_orangemixs.safetensors", "https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt": "orangemix.vae.pt"},"tabs": []}}

    def __init__(self, control_list:List[str]=[]):
        self.controls = control_list

    def load(self, file:str="links.json")  -> None:
        self.file_path = Path(file)

        if not self.file_path.exists():
            self.save()
        else:
            with self.file_path.open('r') as file:
                self.links = json.load(file)

        for ctrl in self.controls:
            if ctrl.tooltip in self.links["ui"]["params"]:
                ctrl.value = self.links["ui"]["params"][ctrl.tooltip]

    # update parameters from ui 
    def update(self)  -> None:
        for ctrl in self.controls:
            self.links["ui"]["params"][ctrl.tooltip] = ctrl.value
            
    # save parameters to file           
    def save(self) -> None:
        self.update()
        with self.file_path.open('w')  as file:
            json.dump(self.links, file, indent=4)

    # get the status of the ressource enabled/cached
    def get_link_status(self, url:str) -> Tuple[bool, bool]:
        enabled = url in self.links["ui"]["enabled"]
        keep = url in self.links["ui"]["keep"]
        return enabled, keep

    # get the ressource name from parameters
    def name_from_link(self , url:str)-> str:
        if 'names' in self.links["ui"] and url in self.links["ui"]['names']:
            return self.links["ui"]['names'][url]
        else:
            return ""
    def link_from_name(self, name: str) -> str:
        if 'names' in self.links["ui"]:
            for url, n in self.links["ui"]['names'].items():
                if n == name:
                    return url
        return ""
    
    # get the ressource name from parameters
    def set_link_name(self , url:str, name:str) -> None:
        if 'names' not in self.links["ui"] :
            self.links["ui"]['names']={}
        self.links["ui"]['names'][url]=name
        
    # update the ressource status
    def update_link_status(self, url:str, to_enable:bool, to_keep:bool) -> None:
        enabled, keep = self.get_link_status(url)
        if to_enable :
            if not enabled :
                self.links["ui"]["enabled"].append(url)
        else :
            if enabled :
                self.links["ui"]["enabled"].remove(url)
                
        if to_keep :
            if not keep :
                self.links["ui"]["keep"].append(url)
        else :
            if keep :
                self.links["ui"]["keep"].remove(url)
        self.save()

    # adds a new ressource
    def add_link(self, url:str, category:str)-> bool:
        def is_valid_url(url):
            parsed = urlparse(url)
            return bool(parsed.scheme) and bool(parsed.netloc)
            
        url=url.strip()
        if is_valid_url(url) and  url not in self.links[category] : 
            self.links[category].append(url)
            self.links["ui"]["enabled"].append(url)
            
            filename=self.links["ui"]["names"].get(url,'')
            if filename =="" :
                 filename=filename_wget(url)     
            if filename :
                self.links["ui"]["names"][url]=filename if "login" not in filename else f"[login error]{url}"
            self.get_accordion(category)
            self.save()
            return True
        return False

        
        
    # removes the specified ressource
    def remove_link(self, category:str, url:str) -> None:
        if url in self.links[category] :
            self.links[category].remove(url)
        if url in self.links["ui"]["enabled"] :
            self.links["ui"]["enabled"].remove(url)
        if url in self.links["ui"]["keep"] :
            self.links["ui"]["keep"].remove(url)
        if url in self.links["ui"]["names"] :
            del self.links["ui"]["names"][url]
        self.get_accordion(category) 
        self.save()


    # create or update the ressource manager 
    def get_accordion(self, category:str, refresh:bool=False) -> Accordion:
        is_new=False
        if category not in self.accordions:
            is_new=True
            self.accordions[category] = Accordion(layout=Layout(width='52%'))
            def on_accordion_change(change):
                if "tabs" not in self.links["ui"] :
                    self.links["ui"]["tabs"]=[]
                
                if change['name'] == 'selected_index' and change['new'] is not None:
                    if category not in self.links["ui"]["tabs"]:
                        self.links["ui"]["tabs"].append(category)
                else :
                    if category in self.links["ui"]["tabs"]:
                        self.links["ui"]["tabs"].remove(category)
            self.accordions[category].observe(on_accordion_change, names='selected_index')

        if (len(self.accordions[category].children)-1) !=len(self.links[category]) or refresh or len(self.links[category])==0:
            vbox_children = []
            vbox_children.append(self.create_add_link(category))
            for url in self.links[category]:
                element_name=self.name_from_link(url)
                if element_name == "" :
                    element_name=url
                hbox = HBox(list(self.create_action_buttons(url, category)) + [Label(value=element_name)])
                vbox_children.append(hbox)
  
            self.accordions[category].children = [VBox(vbox_children)]
    
        self.update_accordion_title(category)
        if is_new :
            if "tabs" in self.links["ui"] and self.links["ui"]["tabs"]:
                if category in self.links["ui"]["tabs"] :
                    self.accordions[category].selected_index = 0
                else:
                    self.accordions[category].selected_index = None

        return self.accordions[category]
    
    # update the count of resources 
    def update_accordion_title(self, category : str) -> None:
        # Display the update button at the end
        keep_cpt=0
        enabled_cpt=0
        for url in self.links[category] :
            if url in self.links["ui"]["keep"] :
                keep_cpt=keep_cpt+1 
            if url in self.links["ui"]["enabled"] :
                enabled_cpt=enabled_cpt+1
        self.accordions[category].set_title(0, str(len(self.links[category])) +" "+ category + " ["+str(enabled_cpt)+"/" +str(keep_cpt) +"]")
        
    #---- ui components ----# 
    # create action buttons for the link manager
    def create_action_buttons(self, url :str , category : str) -> Tuple[ToggleButton, ToggleButton, Button]:
        enabled, keep = self.get_link_status(url)
        
        toggle_enable =ToggleButton(
            layout=Layout(width='90px'),
            value=enabled, 
            description="Enable",
            tooltip="Enable this model",
            button_style=('success' if enabled else ''),
        )
    
        toggle_keep =ToggleButton(
            layout=Layout(width='90px'),
            value=keep, 
            description="Cache",
            tooltip="Cache this model into notebooks",
            button_style=('success' if keep else ''),
        )
    
        delete_button = Button(
            tooltip="Delete this model",
            button_style='danger',     
            layout=Layout(width='30px'),
            icon='trash-o' 
        )
        
        def on_toggle(cc:CallbackDispatcher):
            self.update_link_status(url,toggle_enable.value,toggle_keep.value)
            toggle_keep.button_style = 'success' if toggle_keep.value else ''
            toggle_enable.button_style = 'success' if toggle_enable.value else ''
            self.update_accordion_title(category)
            self.save()
        
        def on_delete(bb:Button):
            self.update_link_status(url, False, False)
            self.remove_link(category,url)
            self.save()

        toggle_enable.observe(on_toggle, 'value')
        toggle_keep.observe(on_toggle, 'value')
        delete_button.on_click(on_delete)
        return toggle_enable , toggle_keep , delete_button

    # create add ressource ui
    def create_add_link(self, category:str)-> HBox:
        link_control = Text(
            value="",
            placeholder=category + " link...",
            layout=Layout(width='80%'),
        )
        
        add_button = Button(
            description='Add',
            layout=Layout(width='20%'),
            icon='plus' 
        )
    
        def on_add(bb:Button) -> None:            
            description='Adding ...'
            add_button.button_style="warning"
            add_button.icon="refresh"
            if self.add_link(link_control.value,category ) :
                add_button.button_style=""
                add_button.icon="plus"
                description='Add'
            else :
                add_button.button_style="danger"
                add_button.icon="plus"
                description='Add'
                
        add_button.on_click(on_add)
        return HBox([link_control, add_button])
        
    # creating output zip/clean tools 
    def create_zip_tool(self) -> HBox:
        #view
        zip_button = Button(
            description='Zip',
            tooltip='Zip folder',
            icon='cube' 
        )
        clean_button = Button(
            description='Clean',
            tooltip='Cleaning folder',
            icon='trash-o'
        )
        zip_clean_button = Button(
            description='Zip & Clean',
            button_style='info', 
            tooltip='Zip folder and clean it up',
            icon='codepen'
        )
            
        def on_zip(bb:Button):
            display("Cleaning..a.")
            bb.description = bb.description.split('>')[0]
            is_cleaning = "clean" in bb.description.lower()
            is_zipping = "zip" in bb.description.lower()
            cleanup_dir = Path(output_control.value)
            if not cleanup_dir.exists():
                display("The folder doesn't exist.")
                bb.button_style = 'danger'  
                bb.icon="times"
                
            if any(cleanup_dir.iterdir()) :
                if is_zipping: 
                    zipped_name = f"{cleanup_dir.name}_{time.strftime('%Y-%m-%d_%H%M')}.zip" 
                    !zip -r {zipped_name} {cleanup_dir}
                    display(f"{zipped_name} zipped!")
                if is_cleaning :
                    folder_name=cleanup_dir.name
                    !rm -rf {cleanup_dir}
                    cleanup_dir.mkdir(exist_ok=True)
                    display(f"{folder_name} cleaned!")
                bb.button_style = 'success' 
                bb.icon="check"
                
            else :
                display("Empty folder.")
                bb.button_style = 'warning' 
        
        #listeners
        zip_button.on_click(on_zip)
        #clean_button.on_click(on_zip)
        zip_clean_button.on_click(on_zip)
    
        #return ui
        return HBox([
            output_control, 
            zip_button, 
            #clean_button,
            zip_clean_button
        ])
           
   
    #---- views ----#
    # ressources manager
    def view_links(self) -> None:
        for category, urls in params.links.items():
            if category != "ui":
                display(HBox([Box(layout=Layout(width='18%')), self.get_accordion(category) ]))

    # output manager 
    def show_zip_tools(self)-> None:
        display(self.create_zip_tool())


    
    @property 
    def options(self) -> Any:
        return self.links["ui"]["params"] 

    @property
    def commandline_arguments(self) -> str:
        arg_list = ["--enable-insecure-extension-access --disable-safe-unpickle --theme dark  --listen --port 6006"]
        if self.options["update_extensions"]  is True:
            arg_list.append("--update-all-extensions")
        if self.options["save_config"]  is True and self.options["config_file"] != "":
            arg_list.extend(["--ui-settings-file", self.options["config_file"]])
        if self.options["save_tasks"]  is True :
            arg_list.extend(["--agent-scheduler-sqlite-file", f"{cached_dir}/tasks.sqlite3"])
        if self.options["extra_cmd"] != "":
            arg_list.extend([x for x in self.options["extra_cmd"].split(" ") if x != ""])
        return " ".join(arg_list)

# css
style = {"description_width": "25%"}
layout = {"width": "70%"}

label_style = HTML(
    "<style> .widget-label, .widget-label-basic >  span { color: white !important; font-weight: bold;}  </style>"
)

# checkboxes
clean_trash_size=get_directory_size('/notebooks/.Trash*')  
clean_trash_control = Checkbox(
    value=True,
    description=f"Clean trash folder [{clean_trash_size}]",
    layout=layout,
    style=style,
    tooltip='clean_trash',
)

cached_config_control = Checkbox(
    value=True,  
    description="Custom config",
    layout=layout,
    style=style,
    tooltip='save_config',
)

cached_tasks_control = Checkbox(
    value=False,  
    description="Persistent task scheduler",
    layout=layout,
    style=style,
    tooltip='save_tasks',
)

cache_size=get_directory_size(f"{cached_dir}")
clean_cached_control = Checkbox(
    value=False,
    description=f"Force re-cache resources [{cache_size}] (removes tasks too)",
    layout=layout,
    style=style,
    tooltip='clean_cached',
)

extn_update_control = Checkbox(
    value=False, 
    description="Update all extensions on launch",
    layout=layout,
    style=style,
    tooltip='update_extensions',
)

sdw_update_control = Checkbox(
    value=False, 
    description="Update stable diffusion first",
    layout=layout,
    style=style,
    tooltip='sdw_update_control',
)

fast_download_control = Checkbox(
    value=False, 
    description="Fast parallel downloads (risky - may cause 403 errors on CivitAI/HuggingFace)",
    layout=layout,
    style=style,
    tooltip='fast_download',
)


commandline_arg_control = Text(
    value="",
    description="Extra commandline arguments",
    style=style,
    layout=layout,
    tooltip='extra_cmd',
)
output_control = Text(
    value="/notebooks/outputs/",
    description="Outputs path",
    placeholder="/notebooks/outputs/",
    style= {"description_width": "46%"},
    layout=Layout(width='38%'),
    tooltip='zip_folder',
)

hf_token_control = Text(
    value=getenv("HF_TOKEN", ""),
    description="HuggingFace token",
    placeholder="hf_... (needed for gated/private models)",
    style=style,
    layout=layout,
    tooltip='hf_token',
)

civitai_token_control = Text(
    value=getenv("CIVITAI_TOKEN", ""),
    description="CivitAI token",
    placeholder="civitai.com -> Account -> API Keys",
    style=style,
    layout=layout,
    tooltip='civitai_token',
)

local_package_control = Text(
    value="sdw_package.tar.xz",
    description="Local package file",
    placeholder="sdw_package.tar.xz (looked up next to this notebook)",
    style=style,
    layout=layout,
    tooltip='local_package',
)

# params instance with all the controls to save
params=Params([clean_trash_control,cached_config_control,cached_tasks_control,clean_cached_control,extn_update_control,sdw_update_control,fast_download_control,output_control,commandline_arg_control,hf_token_control,civitai_token_control,local_package_control])
params.load('links.json')

# autosave: persist checkbox changes to links.json immediately, instead of
# only on the next "Start" run - matches the resource-link toggles below,
# which already save on every click. Text fields (tokens, extra args, output
# path, local package) are excluded - they fire on every keystroke, so
# autosaving them would write the file (and block) on every character typed.
def _autosave_control(_change=None):
    params.save()
for _ctrl in params.controls:
    if isinstance(_ctrl, Checkbox):
        _ctrl.observe(_autosave_control, names='value')

# textboxes
display(
    commandline_arg_control,
    hf_token_control,
    civitai_token_control,
    local_package_control,
)

params.show_zip_tools(),

# checkboxes
display(
    clean_trash_control,
    cached_config_control,
    cached_tasks_control,
    extn_update_control,
    sdw_update_control,
    clean_cached_control,
    fast_download_control,
)

# show ressource manger
params.view_links()

### 2. Start 🚀

In [ ]:
# Run this when you're ready    . 

# Can't run without params
class StopExecution(Exception):
    def _render_traceback_(self):
        pass
try:
    params
except NameError:
    display("Please run the configurations block")
    raise StopExecution

# showing the future link
display(f"Will be ready in a few at https://tensorboard-{getenv('PAPERSPACE_FQDN')} ...")

# read controllers and save parameters
params.update()
params.save()

#cleaning cache
if params.options["clean_cached"]:
    display("Cleaning cached files... " )
    !rm -rf {cached_dir}
    cached_dir.mkdir(parents=True, exist_ok=True)
                
#cleaning trash
if params.options["clean_trash"]:
    trash_size=get_directory_size('/notebooks/.Trash*')
    if trash_size!="0G" and  trash_size!="0B" :
        display("Cleaning trash files... " + trash_size)
        directories = glob.glob('/notebooks/.Trash*')
        for directory in directories:
            try:
                check_output(['rm', '-rf', directory])
                display(f"Deleted: {directory}")  
            except CalledProcessError as e:
                display(f"Error deleting {directory}: {e}")    
            

    
# download repo if needed
config_json = content_dir.joinpath("sdw/config.json")
cached_config_file = notebook_dir.joinpath("config.json")

# seed or update sdw/
sdw_existed_before = sdw_dir.joinpath("launch.py").exists()
if not sdw_existed_before:
    # Prefer a local package file (next to the notebook) over the network -
    # falls back to OFFLINE_PACKAGE_URL if the field is blank or the file isn't there.
    _local_pkg_name = (params.options.get("local_package") or "").strip()
    _local_pkg_path = (notebook_dir / _local_pkg_name) if _local_pkg_name else None
    package_source = _local_pkg_path if (_local_pkg_path and _local_pkg_path.exists()) else OFFLINE_PACKAGE_URL
    seeded = extract_offline_package(package_source, sdw_dir)
    if not seeded:
        print("Cloning AUTOMATIC1111 into sdw/")
        !git clone --quiet --depth 1 -b dev "https://github.com/AUTOMATIC1111/stable-diffusion-webui" {sdw_dir}
elif params.options["sdw_update_control"]:
    print("Updating sdw/")
    !cd {sdw_dir} && git pull
else:
    print("Folder sdw already exists, skipping update.")

# Paperspace's notebook file browser only shows /notebooks - /content (where
# sdw_dir actually lives) isn't visible there, so symlink it in for browsing
# the running install (outputs, models, extensions...) from the Jupyter UI.
inject(sdw_dir, notebook_dir / "sdw", isDirectory=True)



# deal with persistent config - points --ui-settings-file at the notebook
# root. extract_offline_package() already seeded it from the package's bundled
# config.json on first run (see EXTRA_DIR), so it starts with the maintainer's
# tuned settings instead of webui's bare defaults, and stays editable via
# Settings from then on, persisting across instance restarts.
if params.options["save_config"] :
    config_json=cached_config_file
params.options["config_file"]= str(config_json)

# outputs always save to the configured Outputs path - not a toggle, since
# there's no reason to ever want generations dropped in the ephemeral sdw/
# tree instead of the persistent notebook root.
if not output_dir.exists():
    output_dir.mkdir(exist_ok=True)
# config_json may not exist yet on a fresh instance - webui only creates it
# once it actually launches and saves settings, not before. sed on a missing
# file just errors, so skip the redirect for this run; once webui writes the
# file (to the persistent notebook root, since that's where --ui-settings-file
# points), it'll exist for the next run and the redirect applies normally.
if config_json.exists():
    !sed -i 's@"outdir_txt2img_samples": "outputs/txt2img-images"@"outdir_txt2img_samples": "{output_dir}/txt2img-images"@' {config_json}
    !sed -i 's@"outdir_img2img_samples": "outputs/img2img-images"@"outdir_img2img_samples": "{output_dir}/img2img-images"@' {config_json}
    !sed -i 's@"outdir_extras_samples": "outputs/extras-images"@"outdir_extras_samples": "{output_dir}/extras-images"@' {config_json}
    !sed -i 's@"outdir_txt2img_grids": "outputs/txt2img-grids"@"outdir_txt2img_grids": "{output_dir}/txt2img-grids"@' {config_json}
    !sed -i 's@"outdir_img2img_grids": "outputs/img2img-grids"@"outdir_img2img_grids": "{output_dir}/img2img-grids"@' {config_json}
    !sed -i 's@"outdir_save": "log/images"@"outdir_save": "{output_dir}/log/images"@' {config_json}
else:
    display(f"[config] {config_json} doesn't exist yet - webui will create it on first launch; output-path redirect will apply starting next run.")


def split_urls(urls):
    active_urls = []
    persistent_urls = []
    for url in urls:
        enabled, keep = params.get_link_status(url)
        if enabled:
            if not keep:
                if url not in active_urls:
                    active_urls.append(url)
                if url in persistent_urls:
                    persistent_urls.remove(url)
            else:
                if url not in persistent_urls:
                    persistent_urls.append(url)
                if url in active_urls:
                    active_urls.remove(url)
        else:
            if url in active_urls:
                active_urls.remove(url)
            if url in persistent_urls:
                persistent_urls.remove(url)
    return persistent_urls, active_urls


# sync extensions first - webui scans extensions/ once at boot, so these (the
# bundled set from the offline package plus anything added via the UI) need to
# be in place before we launch it below.
ext_urls = params.links.get("extensions", [])
persistent_urls, active_urls = split_urls(ext_urls)
display(f"[extensions] downloading... {(len(persistent_urls)+len(active_urls))} resources")
acquire_checkpoints(persistent_urls, "extensions", True)
acquire_checkpoints(active_urls, "extensions")

# sd-dynamic-prompts isn't wanted - the offline package may still ship it from
# before it was dropped from build_package.py's EXTENSIONS list, so remove it
# outright (folder + its update-on-launch git pull) rather than just disabling
# it in settings.
shutil.rmtree(sdw_dir / "extensions/sd-dynamic-prompts", ignore_errors=True)

# symlinks needed before webui's boot-time extension scan
inject(notebook_dir / "custom/controlnet",   sdw_dir / "extensions/sd-webui-controlnet/models")
inject(notebook_dir / "custom/checkpoints", sdw_dir / "extensions/atlas/checkpoints")

# Paperspace's base image ships a broken wandb install: pytorch_lightning's logger
# registry does `import wandb`, and the installed wandb's generated protobuf code
# doesn't match the installed protobuf version, raising "AttributeError: module
# 'wandb.proto.wandb_internal_pb2' has no attribute 'Result'" deep inside wandb's
# own import chain - which crashes webui's startup before it binds the port.
# webui itself never uses wandb. Removing the package (rather than touching
# protobuf at all) is the safe fix: pytorch_lightning's own wandb.py wraps
# `import wandb` in try/except ModuleNotFoundError specifically for this case, so
# a missing wandb is handled gracefully. (Pinning/upgrading protobuf to fix this
# instead just trades one broken consumer for another - open-clip-torch needs
# protobuf<4, onnxruntime-gpu needs >=4.25.8, those can't both be satisfied, so
# leave protobuf alone entirely.)
!pip uninstall -y -q wandb

# sd-webui-controlnet's preprocessor installer pulls an unpinned, too-new mediapipe
# (0.10.31+), which removed the `mp.solutions` API controlnet_aux's mobile_sam
# preprocessor depends on - "AttributeError: module 'mediapipe' has no attribute
# 'solutions'". Unrelated to protobuf; pin to a version that still has it.
# https://github.com/Mikubill/sd-webui-controlnet/issues/3102
!pip install -q "mediapipe==0.10.21"

environ["GLOBAL_LOG_LEVEL"] = "ERROR"
environ["COMMANDLINE_ARGS"] = params.commandline_arguments
environ["REQS_FILE"] = "requirements_versions.txt"


# --- Setup Variables ---
# Your working Python path (e.g., /usr/local/bin/python3)
working_python_path = sys.executable



# --- Direct Launch Command ---

# 1. Build the full command string:
# [Working Python Executable] [Path to launch.py] [All Command Line Arguments]
# The command is built using the arguments that are already in your 'environ["COMMANDLINE_ARGS"]'
# You may need to add --skip-install if it's not already in params.commandline_arguments
# -u: webui's own stdout isn't a real terminal once piped through subprocess.Popen below,
# so Python's C stdio defaults to block-buffering (4-8KB chunks) instead of line-buffering -
# output appears in silent bursts instead of streaming live. -u forces unbuffered stdout/stderr.
full_command = f"""{working_python_path} -u {sdw_dir}/launch.py {environ["COMMANDLINE_ARGS"]}"""

# models/vaes/loras/embeddings download in a background thread instead of the
# webui launch - so they don't hold up seeing webui's own startup output, and
# don't go silent either: a backgrounded *launch* (subprocess.Popen + cell
# returns) stops streaming to the cell the moment the cell finishes, since
# Jupyter only captures a subprocess's stdout while its cell is still "busy" -
# webui keeps running, but every progress line (model load, CLIP download,
# "Running on local URL...") vanishes with no way to tell it apart from a hang.
def download_remaining_resources():
    for category, urls in params.links.items():
        if category in ("ui", "extensions"):
            continue

        persistent_urls, active_urls = split_urls(urls)

        display(f"[{category}] downloading... {(len(persistent_urls)+len(active_urls))} resources")
        acquire_checkpoints(persistent_urls, category, True)
        acquire_checkpoints(active_urls, category)

    # Lora isn't needed until generation time, so its symlink can happen down here
    inject(notebook_dir / "custom/Lora", sdw_dir / "models/Lora")
    params.save()

import threading
threading.Thread(target=download_remaining_resources, daemon=True).start()

display(f"Launching Stable Diffusion Web UI directly with: {full_command}")
display("Starting 'tensorboard'...")
display(f"Access via this URL: https://tensorboard-{getenv('PAPERSPACE_FQDN')}")

# Foreground (blocking) on purpose - keeps this cell "busy" so webui's own
# startup output streams live instead of disappearing. Stop/interrupt this
# cell to stop webui; the resource downloads above already started in their
# own background thread, so they aren't held up by this blocking.
# Run via subprocess.Popen (instead of the `!` shell magic) so we can rewrite
# Gradio's "Running on local URL: http://0.0.0.0:6006" line - that's just the
# bind address and is meaningless/misleading next to the real public URL
# already shown above - while still streaming every other line live.
def stream_with_url_substitution(cmd: str) -> None:
    public_url = f"https://tensorboard-{getenv('PAPERSPACE_FQDN')}"
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in iter(proc.stdout.readline, ""):
            if "Running on local URL" in line:
                print(f"Running on public URL: {public_url}")
            else:
                print(line, end="")
        proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        raise

stream_with_url_substitution(full_command)

### 2. Other tools

In [ ]:
import shutil
if not (shutil.which("aria2c") and shutil.which("lz4")):
    !apt-get update -y
    !apt-get install -y aria2 lz4
!pip install --upgrade git+https://github.com/huggingface/diffusers.git 
!pip install -U transformers accelerate scipy
!pip install flax==0.5.0 --no-deps
!pip install msgpack rich optax accelerate ftfy scipy
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension
!pip uninstall -y torch torchvision torchaudio
!pip3 install torch==2.2.1 torchvision torchaudio --no-cache-dir --index-url https://download.pytorch.org/whl/cu118